# 进阶实战：CUPED 方差缩减的起死回生效果

## 商业背景
[前司名称] 导购/推荐上线了**常见问题拦截弹窗**，希望降低用户的**核心流失率**。
策略效果极微弱（仅 0.8pp），而用户间天然差异巨大。

**目标**：亲眼见证 CUPED 如何把一个"不显著"的实验捞回"极度显著"。

### 三步降噪组合拳
1. **CUPED**：用实验前数据滤除用户天然噪音
2. **方案 B**：分子分母分别 CUPED + Delta Method
3. **Z-Test**：降噪后计算精准 P-value

---
## 模块 1：数据准备

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(104)

# 常量定义
TOTAL_USERS = 10000
BASELINE_RATE = 0.20         # 基线核心流失率 20%
TRUE_EFFECT = 0.008          # 真实策略效果：仅降低 0.8pp（极微弱！）
USER_NOISE_STD = 0.10        # 用户间天然差异（这是 CUPED 要干掉的噪音源）
RANDOM_NOISE_STD = 0.05      # 纯随机噪声（CUPED 干不掉的）

# 1. 用户天然秉性 (不可观测的 confounder)
# 有的人天生爱找导购/推荐，有的人天生不找 → 构成噪音的主体
user_propensity = np.random.normal(0, USER_NOISE_STD, TOTAL_USERS)

# 2. 实验前 (Pre-Experiment) 用户级核心流失率
rate_pre = BASELINE_RATE + user_propensity + np.random.normal(0, RANDOM_NOISE_STD, TOTAL_USERS)

# 3. 实验期间 (During Experiment)
is_treatment = np.random.binomial(n=1, p=0.5, size=TOTAL_USERS)
rate_exp = (BASELINE_RATE + user_propensity 
            + np.random.normal(0, RANDOM_NOISE_STD, TOTAL_USERS) 
            - is_treatment * TRUE_EFFECT)  # Treatment 组降低 0.8pp

df = pd.DataFrame({
    'group': np.where(is_treatment == 1, 'treatment', 'control'),
    'rate_pre': rate_pre,
    'rate_exp': rate_exp
})

print(f'数据集: {TOTAL_USERS} 用户, 策略效果仅 {TRUE_EFFECT*100}pp')
print(f'用户间天然差异 std={USER_NOISE_STD} >> 策略效果 {TRUE_EFFECT}')
print(f'信噪比极低，策略信号被用户差异彻底淹没！')
df.head()

---
## 模块 2：错误示范 - 不降噪直接检验

In [ ]:
ctrl = df[df['group'] == 'control']['rate_exp']
trt = df[df['group'] == 'treatment']['rate_exp']

stat_naive, p_naive = stats.ttest_ind(trt, ctrl, equal_var=False)

print(f'对照组均值: {ctrl.mean():.4f}')
print(f'实验组均值: {trt.mean():.4f}')
print(f'差异: {trt.mean() - ctrl.mean():.4f}')
print()
print(f'原始 T 检验 P-value: {p_naive:.4f}')
print(f'结论: {"显著" if p_naive < 0.05 else "不显著 -> 金牌策略被错杀！"}')

---
## 模块 3：方案 A - 直接 CUPED 降噪

用实验前的用户级核心流失率作为协变量，扣除用户天然秉性噪音。

$$Y^{cuped}_i = Y^{exp}_i - \theta \times (X^{pre}_i - \overline{X^{pre}})$$

In [ ]:
# Step 1: 计算最优调整系数 theta
cov_matrix = np.cov(df['rate_pre'], df['rate_exp'])
theta = cov_matrix[0, 1] / cov_matrix[0, 0]

# Step 2: CUPED 降噪
mean_pre = df['rate_pre'].mean()
df['cuped_metric'] = df['rate_exp'] - theta * (df['rate_pre'] - mean_pre)

# 方差缩减效果
var_before = df['rate_exp'].var()
var_after = df['cuped_metric'].var()
reduction = (1 - var_after / var_before) * 100

print(f'theta: {theta:.4f}')
print(f'降噪前方差: {var_before:.6f}')
print(f'降噪后方差: {var_after:.6f}')
print(f'方差缩减: {reduction:.1f}%')

In [ ]:
# Step 3: CUPED 后检验
cuped_ctrl = df[df['group'] == 'control']['cuped_metric']
cuped_trt = df[df['group'] == 'treatment']['cuped_metric']

stat_cuped, p_cuped = stats.ttest_ind(cuped_trt, cuped_ctrl, equal_var=False)

print(f'CUPED 降噪后 P-value: {p_cuped:.6f}')
print(f'对比原始 P-value: {p_naive:.4f}')
print()
if p_cuped < 0.05:
    print('CUPED 成功捞回金牌策略！')
else:
    print('仍然不显著')

---
## 模块 4：可视化对比

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图 1: 方差分布对比
axes[0].hist(df['rate_exp'], bins=50, alpha=0.5, label='Original', color='#e74c3c')
axes[0].hist(df['cuped_metric'], bins=50, alpha=0.5, label='CUPED', color='#2ecc71')
axes[0].set_title('Variance Reduction', fontsize=14)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=11)

# 图 2: P-value 对比
labels = ['Naive T-Test', 'CUPED + T-Test']
p_values = [p_naive, p_cuped]
colors = ['#e74c3c' if p > 0.05 else '#2ecc71' for p in p_values]
bars = axes[1].bar(labels, p_values, color=colors, edgecolor='black', linewidth=1.2)
axes[1].axhline(y=0.05, color='black', linestyle='--', linewidth=1.5, label='alpha = 0.05')
axes[1].set_title('P-value Comparison', fontsize=14)
axes[1].set_ylabel('P-value')
axes[1].legend(fontsize=11)
for bar, pv in zip(bars, p_values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                 f'{pv:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 模块 5：方案 B - 分子分母分别 CUPED + Delta Method

当核心指标是 Ratio Metric (核心流失率 = 进线数/访问数) 时，
分子分母各自用历史数据做 CUPED，然后用 Delta Method 组装。

```
Calls_pre  --CUPED-->  Calls_cuped  --+
                                      +-- Delta Method --> SE --> Z-Test
Views_pre  --CUPED-->  Views_cuped  --+
```

In [ ]:
# 为了演示方案 B，从连续指标反向构造 count 数据
np.random.seed(104)

user_prop_b = np.random.gamma(3, 1.0, TOTAL_USERS)

# 实验前
views_pre = np.random.poisson(lam=8 + user_prop_b, size=TOTAL_USERS).clip(1)
calls_pre = np.random.poisson(lam=0.20 * views_pre + user_prop_b * 0.3)

# 实验期
views_exp = np.random.poisson(lam=8 + user_prop_b, size=TOTAL_USERS).clip(1)
calls_exp = np.random.poisson(lam=(0.20 - is_treatment * TRUE_EFFECT) * views_exp + user_prop_b * 0.3)

# 分子 CUPED
theta_c = np.cov(calls_pre, calls_exp)[0,1] / np.var(calls_pre)
calls_cuped = calls_exp - theta_c * (calls_pre - calls_pre.mean())

# 分母 CUPED
theta_v = np.cov(views_pre, views_exp)[0,1] / np.var(views_pre)
views_cuped = views_exp - theta_v * (views_pre - views_pre.mean())

print(f'进线数方差缩减: {(1 - np.var(calls_cuped)/np.var(calls_exp))*100:.1f}%')
print(f'访问数方差缩减: {(1 - np.var(views_cuped)/np.var(views_exp))*100:.1f}%')

In [ ]:
# Delta Method + Z-Test

def delta_se(calls_arr, views_arr):
    n = len(calls_arr)
    mc, mv = calls_arr.mean(), views_arr.mean()
    vc = calls_arr.var(ddof=1)
    vv = views_arr.var(ddof=1)
    cv = np.cov(calls_arr, views_arr)[0, 1]
    R = mc / mv
    var_r = (1/mv**2)*vc/n + (mc**2/mv**4)*vv/n - 2*(mc/mv**3)*cv/n
    return R, np.sqrt(max(var_r, 1e-20))

trt_mask = is_treatment == 1
ctrl_mask = is_treatment == 0

R_trt, se_trt = delta_se(calls_cuped[trt_mask], views_cuped[trt_mask])
R_ctrl, se_ctrl = delta_se(calls_cuped[ctrl_mask], views_cuped[ctrl_mask])

diff_b = R_trt - R_ctrl
se_b = np.sqrt(se_trt**2 + se_ctrl**2)
z_b = diff_b / se_b
p_b = 2 * (1 - stats.norm.cdf(abs(z_b)))

print('=' * 50)
print('终极对比')
print('=' * 50)
print(f'原始 T 检验 (无降噪):        P = {p_naive:.4f}  {"显著" if p_naive < 0.05 else "不显著"}')
print(f'方案 A (CUPED 用户级):        P = {p_cuped:.6f}  {"显著" if p_cuped < 0.05 else "不显著"}')
print(f'方案 B (分别CUPED+Delta):     P = {p_b:.6f}  {"显著" if p_b < 0.05 else "不显著"}')
print('=' * 50)

---
## 结论

| 方案 | P-value | 结论 |
| :--- | :--- | :--- |
| 原始 T 检验 | > 0.05 | 不显著（金牌策略被噪音错杀） |
| 方案 A (CUPED) | < 0.001 | 极度显著（CUPED 起死回生） |
| 方案 B (分别CUPED+Delta) | 对比方案 A | 两者结果接近 |

### 面试话术
> 当用户间天然差异远大于策略效果时（信噪比极低），不做 CUPED 的原始检验会产生假阴性。
> 通过引入实验前的协变量做 CUPED，方差缩减 60%+ 后，微弱的策略信号立刻浮出水面。
> 这就是 CUPED 的核心价值：不是锦上添花，而是起死回生。